# 🤖 Building a Retrieval-Based Chatbot with Text Embeddings

## What You'll Learn

In this notebook, you will learn how to build a **retrieval-based chatbot** — a type of AI assistant that answers questions by finding the most semantically similar question from a knowledge base.

By the end of this tutorial, you'll understand:
- What **text embeddings** are and why they're powerful
- How to use a **pre-trained Sentence Transformer** model
- How **cosine similarity** lets us compare meaning (not just words)
- How to build a simple but effective FAQ chatbot from scratch

---

## 🧠 Conceptual Overview

### What is a Retrieval-Based Chatbot?

Unlike generative chatbots (like GPT), which *generate* new text, a **retrieval-based chatbot** works by:
1. Taking a user's question
2. Comparing it against a set of known questions/answers
3. Returning the best matching answer from that set

This makes it **predictable, fast, and great for FAQ systems** where you know the domain.

### What are Text Embeddings?

An **embedding** is a list of numbers (a vector) that represents the *meaning* of a piece of text. Texts with similar meanings will have vectors that are close to each other in space.

For example:
- `"How do I sign up for Medicare?"` → `[0.12, -0.45, 0.87, ...]`
- `"What is the process to enroll in Medicare?"` → `[0.11, -0.44, 0.85, ...]` ← very close!
- `"What is the weather today?"` → `[-0.63, 0.21, -0.31, ...]` ← far away

This is how the chatbot understands **meaning**, not just keywords.

## 📦 Step 0: Install Dependencies

Before we start, we need to install the required libraries:

- **`sentence-transformers`**: Provides pre-trained models that convert sentences into embeddings
- **`pandas`**: For data manipulation and saving/loading CSVs
- **`scikit-learn`**: Provides the `cosine_similarity` function
- **`numpy`**: For numerical operations on vectors

In [ ]:
# Install required packages
# Run this cell only once — you can skip it if packages are already installed
!pip install sentence-transformers pandas scikit-learn numpy matplotlib

---

# Part 1: Building the Knowledge Base
### `chatbot.py` — Encoding FAQ Questions into Embeddings

## 📖 What This Part Does

This is the **preparation step** of our chatbot. We:
1. Load a powerful pre-trained language model
2. Define our FAQ knowledge base (a list of questions)
3. Convert every question into an embedding vector
4. Save everything to a CSV file so we don't have to re-encode each time

### Why Save to CSV?

Encoding text is computationally expensive. By saving the embeddings once, we can reuse them instantly in future runs without re-loading the model every time.

## 🔧 Imports

- `SentenceTransformer`: The core model class that will encode our text into dense vectors
- `pandas`: We'll use it to organize embeddings into a table and save them as a CSV file

In [ ]:
from sentence_transformers import SentenceTransformer
import pandas as pd

## 🏗️ Loading the Model

We use `all-MiniLM-L6-v2`, a lightweight but powerful model from the Sentence Transformers library.

### Why `all-MiniLM-L6-v2`?

| Property | Value |
|----------|-------|
| Embedding size | 384 dimensions |
| Speed | Very fast |
| Quality | High for semantic similarity tasks |
| Size | ~80MB |

This model was trained to produce embeddings where **semantically similar sentences are close together** in vector space.

> 💡 **Tip:** The first time you run this, the model will be downloaded from HuggingFace Hub and cached locally.

In [ ]:
# Load the pre-trained sentence embedding model
# The model is downloaded automatically from HuggingFace on first use
model = SentenceTransformer("all-MiniLM-L6-v2")
print("✅ Model loaded successfully!")
print(f"   Embedding dimension: {model.get_sentence_embedding_dimension()}")

## 📋 Defining the Knowledge Base

Our knowledge base is a **list of FAQ questions** about Medicare. In a real system, this would come from a database, a spreadsheet, or an API.

Each string here represents a question that our chatbot knows how to handle. The chatbot will match user queries against these questions to find the best answer.

> 🔑 **Key idea:** The more questions you add and the more diverse they are, the more robust your chatbot becomes.

In [ ]:
# Our FAQ knowledge base — a list of questions the chatbot can answer
texts = [
    "How do I get a replacement Medicare card?",
    "What is the monthly premium for Medicare Part B?",
    "How do I terminate my Medicare Part B (medical insurance)?",
    "How do I sign up for Medicare?",
    "Can I sign up for Medicare Part B if I am working and have health insurance through an employer?",
    "How do I sign up for Medicare Part B if I already have Part A?",
    "What are Medicare late enrollment penalties?",
    "What is Medicare and who can get it?",
    "How can I get help with my Medicare Part A and Part B premiums?",
    "What are the different parts of Medicare?",
    "Will my Medicare premiums be higher because of my higher income?",
    "What is TRICARE?",
    "Should I sign up for Medicare Part B if I have Veterans' Benefits?"
]

print(f"📚 Knowledge base: {len(texts)} FAQ questions")
for i, q in enumerate(texts, 1):
    print(f"  {i:2}. {q}")

## 🔢 Encoding Text into Embeddings

Now the magic happens! We pass every question through the model and get back a **384-dimensional vector** for each one.

```
"How do I sign up for Medicare?" 
         │
         ▼  (SentenceTransformer)
[ 0.123, -0.456, 0.789, ..., 0.321 ]  ← 384 numbers
```

The result `embeddings` is a **2D numpy array** of shape **(13 questions × 384 dimensions)**.

In [ ]:
# Encode all FAQ questions into embedding vectors
# Result: numpy array of shape (num_texts, embedding_dim)
embeddings = model.encode(texts)

print(f"✅ Embeddings shape: {embeddings.shape}")
print(f"   → {embeddings.shape[0]} questions × {embeddings.shape[1]} dimensions each")
print(f"\nFirst 5 values of the first embedding:")
print(f"  {embeddings[0][:5]}")

## 💾 Saving Embeddings to CSV

We create a pandas DataFrame where:
- **Columns 0 to 383** hold the 384 embedding values
- **The `sentence` column** holds the original text

This CSV acts as our **persistent vector database** — simple but effective for caching embeddings.

> 💡 **In production systems**, you'd use a proper vector database like Pinecone, Chroma, or FAISS. But CSV works great for learning!

In [ ]:
# Build a DataFrame: numeric embedding columns + original sentence column
df = pd.DataFrame(embeddings)
df["sentence"] = texts

print("📊 DataFrame shape:", df.shape)
print(f"   {df.shape[1] - 1} embedding dimensions + 1 sentence column")
print("\nPreview (first 3 rows, columns 0-4 + sentence):")
print(df[[0, 1, 2, 3, 4, "sentence"]].head(3).to_string())

# Save to CSV for reuse later
df.to_csv("embeddings.csv", index=False)
print("\n✅ Saved to embeddings.csv")

---

# Part 2: The Chatbot — Matching User Queries
### `embedding.py` — Semantic Search with Cosine Similarity

## 📖 What This Part Does

This is the **inference step** — where the actual chatbot lives. Given a user's question:
1. We load our pre-built knowledge base from the CSV
2. We encode the user's question into a vector
3. We compute how similar the user's vector is to each FAQ vector
4. We return the most similar FAQ question

## 🔍 Cosine Similarity — The Core Algorithm

**Cosine similarity** measures the angle between two vectors:

| Score | Meaning |
|-------|---------|
| `1.0` | Identical meaning |
| `0.7` | Closely related |
| `0.4` | Loosely related |
| `0.0` | Completely unrelated |

Unlike Euclidean distance, cosine similarity is scale-invariant — it only cares about the **direction** of the vector, not its magnitude. This makes it ideal for comparing text embeddings.

In [ ]:
# New imports for the matching engine
from sklearn.metrics.pairwise import cosine_similarity  # Computes vector similarity
import numpy as np                                       # np.argmax to find best match

## 📂 Loading the Pre-Built Knowledge Base

Instead of re-encoding all FAQ questions, we load the saved CSV. We:
- Extract the `sentence` column → original text labels for display
- Drop `sentence` and keep the rest → pure numeric embedding matrix for math

This separation is important: we need vectors for cosine similarity, and text for the final answer.

In [ ]:
# Load the pre-built embeddings from disk
df = pd.read_csv("embeddings.csv")

# Separate: text labels vs numeric embedding matrix
sentences = df["sentence"].tolist()                    # List[str] → display labels
embeddings = df.drop(columns=["sentence"]).values      # ndarray (13, 384) → for math

print(f"✅ Loaded {len(sentences)} FAQs")
print(f"   Embedding matrix: {embeddings.shape}")

## ❓ User Query

This is what the user types. The query **doesn't need to match any FAQ word-for-word** — embeddings understand semantic meaning.

Try modifying this query to see how the chatbot responds to different phrasings!

In [ ]:
# Try changing this to test semantic understanding!
query = "How can Medicare help me?"
print(f"🗣️  User: '{query}'")

## ⚡ Encoding the Query

We pass the user's question through the same model used to build the knowledge base. It's critical to use the **same model** — only then will the vector spaces align and comparisons be meaningful.

> ⚠️ We wrap the query in `[query]` because `encode()` expects a list. Output shape: `(1, 384)`.

In [ ]:
# Encode the user query — must use the SAME model as the knowledge base!
query_embedding = model.encode([query])   # Shape: (1, 384)

print(f"Query vector shape: {query_embedding.shape}")
print(f"First 5 values: {query_embedding[0][:5].round(4)}")

## 📐 Computing Cosine Similarity

`cosine_similarity(query_embedding, embeddings)` computes the dot-product similarity between our 1 query vector and all 13 FAQ vectors simultaneously.

Result shape: `(1, 13)` → we take `[0]` to get a flat array of 13 scores.

Then `np.argmax()` gives us the **index** of the highest score — our best match!

In [ ]:
# Compute similarity between query and ALL FAQ vectors at once
# Shape: (1, 13) → [0] flattens to (13,)
similarities = cosine_similarity(query_embedding, embeddings)[0]

# Display a ranked similarity table
print(f"Similarity scores for: '{query}'")
print("-" * 65)
ranked = sorted(zip(similarities, sentences), reverse=True)
for score, sentence in ranked:
    bar = "█" * int(score * 25)
    label = sentence[:45] + "..." if len(sentence) > 45 else sentence
    print(f"  {score:.4f}  {bar:25s}  {label}")

## 🏆 Best Match — The Chatbot's Answer

`np.argmax(similarities)` returns the index of the FAQ with the highest cosine similarity to our query. That's our answer!

We also add a simple **confidence interpretation** based on the score.

In [ ]:
# Index of best-matching FAQ
best_match_idx = np.argmax(similarities)
score = similarities[best_match_idx]

print("=" * 60)
print("🤖 CHATBOT RESPONSE")
print("=" * 60)
print(f"  Query:      '{query}'")
print(f"  Best match: '{sentences[best_match_idx]}'")
print(f"  Score:       {score:.4f}")
print("=" * 60)

if score > 0.85:
    print("  ✅ Very high confidence")
elif score > 0.65:
    print("  ⚠️  Moderate confidence")
else:
    print("  ❌ Low confidence — may be out of scope")

---

# Part 3: Reusable Chatbot Function

Let's wrap everything into a clean, reusable `chatbot()` function with a confidence threshold — if no FAQ is similar enough, the bot admits it doesn't know.

In [ ]:
def chatbot(user_query: str, threshold: float = 0.4) -> dict:
    """
    Retrieval-based chatbot using semantic similarity.

    Args:
        user_query: The user's natural language question.
        threshold:  Minimum similarity to return a match (default 0.4).

    Returns:
        dict with 'query', 'best_match', and 'score'.
    """
    query_vec = model.encode([user_query])
    scores = cosine_similarity(query_vec, embeddings)[0]
    best_idx = np.argmax(scores)
    best_score = float(scores[best_idx])

    if best_score < threshold:
        return {"query": user_query,
                "best_match": "I'm sorry, I couldn't find a relevant answer.",
                "score": best_score}

    return {"query": user_query,
            "best_match": sentences[best_idx],
            "score": best_score}


# --- Test with various queries ---
test_queries = [
    "How can Medicare help me?",
    "What will I pay each month for Medicare?",
    "I work full time, can I still get Medicare?",
    "How do I get a new Medicare card if I lost mine?",
    "What is the best pizza recipe?"   # Out of scope!
]

print("=" * 65)
for q in test_queries:
    r = chatbot(q)
    print(f"Q: {r['query']}")
    print(f"A: {r['best_match']}  [{r['score']:.4f}]")
    print("-" * 65)

---

# Part 4: Visualizing the Embedding Space

## 🗺️ Seeing Semantic Clusters

Embeddings live in 384-dimensional space — we can't visualize that directly. But **PCA (Principal Component Analysis)** projects them to 2D while preserving their relative distances.

Questions about similar topics should appear **closer together** on the plot. This lets us visually inspect whether our embedding model understands semantic relationships correctly.

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

# Reduce 384D → 2D using PCA
pca = PCA(n_components=2)
reduced = pca.fit_transform(embeddings)

fig, ax = plt.subplots(figsize=(13, 7))
ax.scatter(reduced[:, 0], reduced[:, 1], c='steelblue', s=120, zorder=3)

for i, sent in enumerate(sentences):
    label = sent[:38] + "..." if len(sent) > 38 else sent
    ax.annotate(label, (reduced[i, 0], reduced[i, 1]),
                textcoords="offset points", xytext=(8, 4), fontsize=8)

ax.set_title("2D PCA of FAQ Embeddings — Closer = More Similar Meaning", fontsize=13)
ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Variance explained: {sum(pca.explained_variance_ratio_)*100:.1f}%")

---

# 🎓 Summary & Key Takeaways

You've built a working semantic chatbot! Here's what you learned:

| Concept | What it Does |
|---------|-------------|
| **Text Embedding** | Converts text into a numerical vector that captures meaning |
| **SentenceTransformer** | Pre-trained model for high-quality sentence embeddings |
| **Cosine Similarity** | Measures semantic closeness between two vectors |
| **Retrieval-Based QA** | Finds the best existing answer rather than generating a new one |

### Architecture

```
chatbot.py        → Build & cache the knowledge base (run once)
embedding.py      → Query the knowledge base at runtime
embeddings.csv    → Persistent vector store
```

### Ideas to Extend This Project

1. **Add actual answers** — Store answer text alongside questions and return them
2. **Expand the knowledge base** — Add hundreds of FAQ pairs from real documentation
3. **Use a vector database** — Replace the CSV with FAISS, Chroma, or Pinecone
4. **Add a web UI** — Wrap in Streamlit or Flask for a browser interface
5. **Hybrid search** — Combine semantic search with keyword search (BM25)

### Resources

- [Sentence Transformers Documentation](https://www.sbert.net/)
- [HuggingFace: all-MiniLM-L6-v2](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2)
- [Scikit-learn: cosine_similarity](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.pairwise.cosine_similarity.html)
- [Illustrated Word2Vec — Jay Alammar](https://jalammar.github.io/illustrated-word2vec/)